|                |   |
:----------------|---|
| **Nombre**     Elisa Aguirre|   |
| **Fecha**      27/04/25|   |
| **Expediente** 738894 |   |

# A08 - Bootstrapping + Aggregating

In [2]:
import pandas as pd 
import numpy as np

In [3]:
df=pd.read_excel(r"C:\Users\elisa\Downloads\Motor Trend Car Road Tests.xlsx")
df.head()

,model,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


## Regresión lineal

En esta parte se cargó el dataset **Motor Trend Car Road Tests** y se seleccionaron las variables que se usarán en el modelo. La variable respuesta es `mpg`, que representa el rendimiento del automóvil, mientras que las variables explicativas son `hp` y `qsec`.

El modelo que se ajusta es:

$$
mpg = \beta_0 + \beta_1 hp + \beta_2 qsec
$$

Después se utilizó `LinearRegression()` para ajustar la regresión lineal y obtener los coeficientes estimados del modelo: el intercepto $\beta_0$, el coeficiente de `hp` y el coeficiente de `qsec`.

In [4]:
X = df[['hp', 'qsec']]
y = df['mpg']

In [5]:
from sklearn.linear_model import LinearRegression
modelo = LinearRegression()

In [6]:
modelo.fit(X, y)

LinearRegression()

In [7]:
print("Intercepto (beta0):", modelo.intercept_)
print("Beta hp:", modelo.coef_[0])
print("Beta qsec:", modelo.coef_[1])

Intercepto (beta0): 48.32370516913445
Beta hp: -0.08459304367409272
Beta qsec: -0.8865796246342723


## Intervalos de confianza de los betas

Para calcular los intervalos de confianza al 95% de los coeficientes se utilizó `statsmodels`, ya que esta librería permite obtener información estadística del modelo, como errores estándar, valores p e intervalos de confianza.

Primero se agregó una constante a la matriz de variables explicativas para incluir el intercepto del modelo. Después se ajustó el modelo OLS y se obtuvieron los intervalos de confianza con `conf_int(alpha = 0.05)`.

El nivel de significancia usado fue:

$$
\alpha = 0.05
$$

por lo que el nivel de confianza es del 95%.

In [8]:
import statsmodels.api as sm
X = sm.add_constant(X)

# Ajustar modelo
modelo_sm = sm.OLS(y, X).fit()

# Ver resumen completo
print(modelo_sm.summary())

                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.637
Model:                            OLS   Adj. R-squared:                  0.612
Method:                 Least Squares   F-statistic:                     25.43
Date:                Mon, 04 May 2026   Prob (F-statistic):           4.18e-07
Time:                        16:43:22   Log-Likelihood:                -86.170
No. Observations:                  32   AIC:                             178.3
Df Residuals:                      29   BIC:                             182.7
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         48.3237     11.103      4.352      0.0

In [9]:
intervalos = modelo_sm.conf_int(alpha=0.05)
intervalos.columns = ['LI 95%', 'LS 95%']

intervalos

,LI 95%,LS 95%
const,25.614894,71.032516
hp,-0.113089,-0.056097
qsec,-1.979929,0.206770


## Bootstrap

En esta parte se aplicó el método bootstrap para analizar la estabilidad de los coeficientes del modelo. Primero se definió:

$$
m = n
$$

donde $m$ es el número de observaciones del dataset original.

Después se crearon 1000 muestras bootstrap. Cada muestra tiene el mismo tamaño que el dataset original, pero se construye seleccionando observaciones con reemplazo. Esto significa que una misma observación puede aparecer más de una vez dentro de una muestra.

Para hacerlo sin usar una función directa de bootstrap, se generaron índices aleatorios con `np.random.choice()` y después se seleccionaron las filas correspondientes con `iloc`.

In [10]:
import numpy as np

m = len(df)

muestras_bootstrap = []

for i in range(1000):
    indices = np.random.choice(m, size=m, replace=True)
    muestra = df.iloc[indices]
    muestras_bootstrap.append(muestra)

In [11]:
muestras_bootstrap[0].head()

,model,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
25,Fiat X1-9,27.3,4,79.0,66,4.08,1.935,18.90,1,1,4,1
11,Merc 450SE,16.4,8,275.8,180,3.07,4.070,17.40,0,0,3,3
15,Lincoln Continental,10.4,8,460.0,215,3.00,5.424,17.82,0,0,3,4
15,Lincoln Continental,10.4,8,460.0,215,3.00,5.424,17.82,0,0,3,4
19,Toyota Corolla,33.9,4,71.1,65,4.22,1.835,19.90,1,1,4,1


## Regresión lineal en cada muestra bootstrap

Después de crear las 1000 muestras bootstrap, se ajustó una regresión lineal en cada una de ellas usando nuevamente `mpg` como variable respuesta y `hp` y `qsec` como variables explicativas.

En cada iteración se guardaron los coeficientes estimados:

$$
\beta_0,\ \beta_{hp},\ \beta_{qsec}
$$

Al final se construyó un dataframe llamado `betas_bootstrap`, donde cada fila representa los coeficientes obtenidos en una muestra bootstrap.

In [12]:
betas_bootstrap = []

for muestra in muestras_bootstrap:
    
    X_boot = muestra[['hp', 'qsec']]
    y_boot = muestra['mpg']
    
    modelo = LinearRegression()
    modelo.fit(X_boot, y_boot)
    
    betas_bootstrap.append([
        modelo.intercept_,
        modelo.coef_[0],
        modelo.coef_[1]
    ])

betas_bootstrap = pd.DataFrame(
    betas_bootstrap,
    columns=['beta0', 'beta_hp', 'beta_qsec']
)

betas_bootstrap.head()

,beta0,beta_hp,beta_qsec
0,41.421056,-0.088363,-0.491552
1,59.047855,-0.090861,-1.507595
2,62.826880,-0.128352,-1.339796
3,39.227088,-0.088304,-0.413763
4,60.316163,-0.100686,-1.459499


## Media, desviación estándar e intervalos bootstrap

Con los coeficientes obtenidos de las 1000 muestras bootstrap se calculó la media y la desviación estándar de cada beta.

La media bootstrap permite observar el valor promedio estimado de cada coeficiente, mientras que la desviación estándar mide la variabilidad de los coeficientes entre las diferentes muestras.

Los intervalos de confianza bootstrap al 95% se calcularon con:

$$
IC = \bar{\beta}_{boot} \pm 1.96(SE_{boot})
$$

Se utiliza 1.96 porque corresponde al valor crítico de la distribución normal estándar para un nivel de confianza del 95%.

In [13]:
desv_betas = betas_bootstrap.std()

desv_betas


beta0        10.739186
beta_hp       0.016953
beta_qsec     0.514668
dtype: float64

In [14]:
media_betas = betas_bootstrap.mean()

media_betas

beta0        50.145998
beta_hp      -0.088224
beta_qsec    -0.965254
dtype: float64

In [15]:
tabla_bootstrap = pd.DataFrame({
    'Media beta bootstrap': media_betas,
    'Desviación estándar': desv_betas,
    'LI 95%': media_betas - 1.96 * desv_betas,
    'LS 95%': media_betas + 1.96 * desv_betas
})

tabla_bootstrap

,Media beta bootstrap,Desviación estándar,LI 95%,LS 95%
beta0,50.145998,10.739186,29.097193,71.194804
beta_hp,-0.088224,0.016953,-0.121451,-0.054997
beta_qsec,-0.965254,0.514668,-1.974003,0.043496


## Comparación de resultados

Finalmente, se compararon los betas originales con las medias bootstrap y sus intervalos de confianza.

El coeficiente de `hp` se mantiene relativamente cercano entre el modelo original y el bootstrap, además de que su intervalo de confianza no incluye el cero. Esto sugiere que `hp` es una variable significativa para explicar `mpg`.

Por otro lado, el coeficiente de `qsec` presenta mayor variabilidad y su intervalo de confianza incluye el cero, por lo que no hay suficiente evidencia para considerarlo significativo dentro del modelo.

En general, el bootstrap permite evaluar la estabilidad de los coeficientes y observar qué tan sensibles son las estimaciones al tomar diferentes muestras con reemplazo del dataset original.

In [16]:
betas_originales = pd.Series({
    'beta0': modelo.intercept_,
    'beta_hp': modelo.coef_[0],
    'beta_qsec': modelo.coef_[1]
})

tabla_comparacion = pd.DataFrame({
    'Beta original': betas_originales,
    'Media bootstrap': media_betas,
    'Desv. estándar bootstrap': desv_betas,
    'LI 95% bootstrap': media_betas - 1.96 * desv_betas,
    'LS 95% bootstrap': media_betas + 1.96 * desv_betas
})

tabla_comparacion

,Beta original,Media bootstrap,Desv. estándar bootstrap,LI 95% bootstrap,LS 95% bootstrap
beta0,40.222076,50.145998,10.739186,29.097193,71.194804
beta_hp,-0.063323,-0.088224,0.016953,-0.121451,-0.054997
beta_qsec,-0.624053,-0.965254,0.514668,-1.974003,0.043496


In [17]:
intervalos

,LI 95%,LS 95%
const,25.614894,71.032516
hp,-0.113089,-0.056097
qsec,-1.979929,0.206770


# Train-Test

In [18]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['mpg', 'model'])
y = df['mpg']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, random_state=42)

In [19]:
columnas = X.columns
resultados = []
for i in range(1000):
    vars_random = np.random.choice(columnas, size=3, replace=False)
    Xi_train = X_train[list(vars_random)]
    modelo = LinearRegression()
    modelo.fit(Xi_train, y_train)
    resultados.append({
        'variables': vars_random,
        'modelo': modelo
    })


In [20]:
predicciones = []

for res in resultados:
    
    vars_modelo = res['variables']
    modelo = res['modelo']
    
    Xi_test = X_test[list(vars_modelo)]
    
    y_pred = modelo.predict(Xi_test)
    
    predicciones.append(y_pred)

predicciones = np.array(predicciones)

predicciones.shape

(1000, 16)

In [21]:
y_pred_promedio = predicciones.mean(axis=0)
y_pred_promedio

array([20.38050184, 10.19144109, 14.38940544, 27.12394849, 23.5870039 ,
       20.25480098, 13.63148215, 27.46499152, 15.32050304, 21.81429188,
       15.42244312, 10.42812668, 19.75972014, 15.28602886, 14.7826312 ,
       13.515361  ])

In [22]:
from sklearn.metrics import r2_score
r2_promedio = r2_score(y_test, y_pred_promedio)
r2_promedio

0.7825000741352736

In [23]:
resultados[37]["modelo"].predict(X_test[resultados[37]["variables"]])

array([21.25218278,  8.45892942, 13.61737785, 27.54338718, 22.03968463,
       21.01431241, 12.72646315, 27.81476041, 17.70334681, 22.26526482,
       15.38268845,  9.53374751, 20.76718695, 17.50369716, 17.80317164,
       14.47524499])

In [24]:
resultados_df = pd.DataFrame(resultados)

resultados_df.head()

,variables,modelo
0,"[am, disp, carb]",LinearRegression()
1,"[carb, wt, disp]",LinearRegression()
2,"[qsec, wt, vs]",LinearRegression()
3,"[drat, wt, hp]",LinearRegression()
4,"[hp, carb, am]",LinearRegression()


## Random forest

In [53]:
from sklearn.model_selection import KFold, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


X = df.drop(columns=['mpg', 'model'])
y = df['mpg']

kfold = KFold(n_splits=10, shuffle=True, random_state=42)

rf = RandomForestRegressor(random_state=42)
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestRegressor(random_state=42))
])
param_dist = {
    'n_estimators': [200, 300, 500, 800],
    'max_depth': [None, 3, 4, 5, 6, 8],
    'max_leaf_nodes': [None, 10, 15, 20, 30],
}

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=50,
    cv=kfold,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X, y)

RandomizedSearchCV(cv=KFold(n_splits=10, random_state=42, shuffle=True),
                   estimator=RandomForestRegressor(random_state=42), n_iter=50,
                   n_jobs=-1,
                   param_distributions={'max_depth': [None, 3, 4, 5, 6, 8],
                                        'max_leaf_nodes': [None, 10, 15, 20,
                                                           30],
                                        'n_estimators': [200, 300, 500, 800]},
                   random_state=42, scoring='r2')

In [54]:
random_search.best_score_

np.float64(0.5116072603953695)

In [55]:
mejor_rf = random_search.best_estimator_

In [56]:
y_pred_total = mejor_rf.predict(X)

In [57]:
from sklearn.metrics import r2_score

r2_total = r2_score(y, y_pred_total)

r2_total

0.9723145983864327

## Random Forest con K-Folds y optimización bayesiana

En esta parte se ajustó un modelo de Random Forest Regressor para predecir `mpg` utilizando las demás variables del dataset como variables explicativas.

Primero se definió una validación cruzada con `KFold`, usando `k = 10`. Esto significa que el dataset se divide en 10 partes; en cada iteración se entrena el modelo con 9 partes y se evalúa con la parte restante. La métrica utilizada para evaluar el desempeño fue el coeficiente de determinación \(R^2\).

Después se aplicó optimización bayesiana mediante `BayesSearchCV` para encontrar la mejor combinación de hiperparámetros del modelo. Los hiperparámetros optimizados fueron el número de árboles (`n_estimators`), la profundidad máxima de los árboles (`max_depth`) y el número máximo de hojas (`max_leaf_nodes`).

Finalmente, se seleccionó el modelo con el mayor \(R^2\) promedio obtenido en la validación cruzada.

In [65]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV, KFold

boost = GradientBoostingRegressor(random_state=42)

param_dist = {
    'n_estimators': [800, 1000, 1500],
    'learning_rate': [0.01, 0.005],
    'max_depth': [4, 5, 6],
    'max_leaf_nodes': [None, 30, 50, 80]
}

kfold = KFold(n_splits=10, shuffle=True, random_state=42)

boost_search = RandomizedSearchCV(
    estimator=boost,
    param_distributions=param_dist,
    n_iter=60,
    cv=kfold,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)

boost_search.fit(X, y)

RandomizedSearchCV(cv=KFold(n_splits=10, random_state=42, shuffle=True),
                   estimator=GradientBoostingRegressor(random_state=42),
                   n_iter=60, n_jobs=-1,
                   param_distributions={'learning_rate': [0.01, 0.005],
                                        'max_depth': [4, 5, 6],
                                        'max_leaf_nodes': [None, 30, 50, 80],
                                        'n_estimators': [800, 1000, 1500]},
                   random_state=42, scoring='r2')

In [66]:
boost_search.best_score_
boost_search.best_params_

{'n_estimators': 800,
 'max_leaf_nodes': None,
 'max_depth': 4,
 'learning_rate': 0.005}

In [67]:
mejor_boost = boost_search.best_estimator_

y_pred_total = mejor_boost.predict(X)

from sklearn.metrics import r2_score
r2_total = r2_score(y, y_pred_total)

r2_total

0.998924201819557